# Chapter 16 — DPO from Scratch## Direct Preference Optimization[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**GPU recommended. Runtime: ~20 minutes.**Derives and implements the DPO loss from first principles, then compares with PPO on a toy alignment task.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
print('Ready')

## 16.1 The DPO Derivation in 5 Steps**Step 1**: RLHF objective: max E[r(x,y)] - β·KL(π||π_ref)**Step 2**: Closed-form optimal policy: π*(y|x) = π_ref(y|x)·exp(r/β) / Z(x)**Step 3**: Invert to express reward: r(x,y) = β·log[π*(y|x)/π_ref(y|x)] + β·log Z(x)**Step 4**: Plug into Bradley-Terry: P(y_w≻y_l) = σ(r_w - r_l) = σ(β·log[π(y_w)/π_ref(y_w)] - β·log[π(y_l)/π_ref(y_l)])**Step 5**: Maximise likelihood → **DPO loss**

In [ ]:
def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    beta: float = 0.1,
    length_normalise: bool = True,
    chosen_lengths: torch.Tensor = None,
    rejected_lengths: torch.Tensor = None,
) -> tuple:
    """
    DPO loss with optional length normalisation.
    Returns (loss, reward_accuracy, reward_margin, chosen_rewards, rejected_rewards)
    """
    if length_normalise and chosen_lengths is not None:
        policy_chosen_logps   = policy_chosen_logps   / chosen_lengths
        policy_rejected_logps = policy_rejected_logps / rejected_lengths
        ref_chosen_logps      = ref_chosen_logps      / chosen_lengths
        ref_rejected_logps    = ref_rejected_logps    / rejected_lengths
    
    pi_logratios_chosen   = policy_chosen_logps   - ref_chosen_logps
    pi_logratios_rejected = policy_rejected_logps - ref_rejected_logps
    
    logits = beta * (pi_logratios_chosen - pi_logratios_rejected)
    loss   = -F.logsigmoid(logits).mean()
    
    chosen_rewards   = (beta * pi_logratios_chosen).detach()
    rejected_rewards = (beta * pi_logratios_rejected).detach()
    accuracy = (chosen_rewards > rejected_rewards).float().mean()
    margin   = (chosen_rewards - rejected_rewards).mean()
    
    return loss, accuracy, margin, chosen_rewards, rejected_rewards

print('DPO loss function defined.')

## 16.2 Visualise How DPO Trains

In [ ]:
# Simulate DPO training on synthetic log-probabilities
np.random.seed(42)
torch.manual_seed(42)

batch_size, seq_len, vocab = 32, 64, 50000

# Simulate a simple policy being trained
chosen_logp   = torch.randn(batch_size) * 0.3 - 4.0  # initial: low log-probs
rejected_logp = torch.randn(batch_size) * 0.3 - 4.0
ref_chosen    = torch.randn(batch_size) * 0.2 - 4.2
ref_rejected  = torch.randn(batch_size) * 0.2 - 3.8

# Track what happens as we update chosen_logp upward, rejected_logp downward
steps = 100
losses, accuracies, margins = [], [], []

for step in range(steps):
    # Simulate gradient step: chosen logp increases, rejected decreases
    chosen_logp   = chosen_logp   + 0.02 * torch.randn(batch_size).abs()
    rejected_logp = rejected_logp - 0.015 * torch.randn(batch_size).abs()
    
    loss, acc, margin, cr, rr = dpo_loss(chosen_logp, rejected_logp, ref_chosen, ref_rejected)
    losses.append(loss.item())
    accuracies.append(acc.item())
    margins.append(margin.item())

fig, axes = plt.subplots(1, 3, figsize=(14,4))
axes[0].plot(losses, color='tomato', linewidth=2)
axes[0].set_title('DPO Loss'); axes[0].set_xlabel('Step'); axes[0].grid(True, alpha=0.3)
axes[1].plot(accuracies, color='steelblue', linewidth=2)
axes[1].set_title('Reward Accuracy'); axes[1].set_xlabel('Step'); axes[1].grid(True, alpha=0.3)
axes[2].plot(margins, color='seagreen', linewidth=2)
axes[2].set_title('Reward Margin'); axes[2].set_xlabel('Step'); axes[2].grid(True, alpha=0.3)
plt.suptitle('DPO Training Dynamics (Simulated)')
plt.tight_layout(); plt.show()

## 16.3 β Sensitivity Analysis

In [ ]:
betas = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
results = []

for beta in betas:
    # Simulate what different β values do to the reward gap
    gap = torch.linspace(-3, 3, 100)
    prob_prefer = torch.sigmoid(beta * gap)
    results.append(prob_prefer.numpy())

fig, ax = plt.subplots(figsize=(10,5))
gap_np = torch.linspace(-3, 3, 100).numpy()
colors = plt.cm.viridis(np.linspace(0, 1, len(betas)))
for beta, res, color in zip(betas, results, colors):
    ax.plot(gap_np, res, label=f'β={beta}', color=color, linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Log-ratio gap (chosen - rejected)')
ax.set_ylabel('P(chosen preferred)')
ax.set_title('Effect of β on DPO Preference Probability')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('Small β = aggressive reward shaping')
print('Large β = conservative, stays close to reference')

## ✅ Key Takeaways1. DPO eliminates the reward model by solving RLHF in closed form2. β controls how aggressively the policy diverges from the reference (same role as KL penalty in PPO-RLHF)3. Length normalisation prevents the model from preferring shorter completions4. Reward accuracy and margin are the key training diagnostics to track